In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
DIFFUSION_CONFIG_PATH = "config/diffusion_full_imagenet64_flow_uncond.yaml"   # relative to repo root
DIFFUSION_CKPT_PATH = None   # None -> load best_ckpt.pt from experiment output dir
UNCONDITIONAL = True    # True -> skip VAE / precomputed latents, pass z=0
USE_EMA_WEIGHTS = True   # sample from the checkpoint's EMA shadow (models/ema.py) if
                         # present, falling back to the raw weights with a warning if the
                         # checkpoint predates ema_decay being set during training

# FID sampling budget. The real reference side is capped at whatever exists in the val
# split (never duplicated to pad it out -- that would understate real-distribution
# variance); the generated side always produces exactly N_FID_SAMPLES images. FID bias
# and variance both drop as this grows, but so does wall-clock time, since each image
# costs NUM_INFERENCE_STEPS model forward passes -- start small to sanity-check the
# pipeline, then scale up.
N_FID_SAMPLES = 2048
FID_BATCH_SIZE = 64        # samples generated per model call -- tune to GPU memory

GUIDANCE_SCALE = 1.0    # > 1.0 requires a CFG-trained model (cfg_uncond_prob > 0)
SEED = 42

# DDPM-only options (ignored when noise_process is 'flow' or 'vpsde')
SAMPLER = "ddpm"   # 'ddpm' or 'ddim'
ETA = 0.0          # ddim only: 0=deterministic, 1~ddpm stochasticity

# Number of steps for ddim / flow / vpsde (ddpm always uses all T steps)
NUM_INFERENCE_STEPS = 100

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)

## Imports

In [ ]:
import os
import sys
import types
import yaml
import random
import itertools
import numpy as np
import subprocess
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import linalg
from tqdm.auto import tqdm

sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from models.beta_vae import BetaVAE
from models.unet import UNet
from models.noise_process import DDPM, RectifiedFlow, VPSDE
from utils import make_noise_schedule, diffusion_sample, flow_sample, vpsde_sample, make_run_tag
from data_utils import build_dataloaders, build_imagenet64_val_dataloader, build_encode_dataloader
from torchvision.models import inception_v3, Inception_V3_Weights

## Load models

In [ ]:
with open(os.path.join(REPO_ROOT, DIFFUSION_CONFIG_PATH)) as f:
    diff_cfg = types.SimpleNamespace(**yaml.safe_load(f))

diff_cfg.output_dir = os.path.join(REPO_ROOT, f"{diff_cfg.output_dir.lstrip('./')}_{make_run_tag(diff_cfg)}")
# betavae paths are absent in unconditional configs and in precomputed-latents configs
# (keys commented out in yaml)
if hasattr(diff_cfg, 'betavae_config_path'):
    diff_cfg.betavae_config_path = os.path.join(REPO_ROOT, diff_cfg.betavae_config_path.lstrip("./"))
if hasattr(diff_cfg, 'betavae_ckpt_path'):
    diff_cfg.betavae_ckpt_path   = os.path.join(REPO_ROOT, diff_cfg.betavae_ckpt_path.lstrip("./"))
if not os.path.isabs(diff_cfg.data_dir):
    diff_cfg.data_dir = os.path.join(REPO_ROOT, diff_cfg.data_dir.lstrip("./"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- frozen beta-VAE, only loaded when the diffusion model actually conditions on one ---
if UNCONDITIONAL:
    vae = None
    print("Unconditional mode: skipping beta-VAE, z is a zero vector")
elif hasattr(diff_cfg, 'betavae_config_path'):
    with open(diff_cfg.betavae_config_path) as f:
        vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))
    vae = BetaVAE(
        input_channels=vae_cfg.input_channels,
        input_height=vae_cfg.input_height,
        input_width=vae_cfg.input_width,
        latent_dim=vae_cfg.latent_dim,
        hidden_dim=vae_cfg.hidden_dim,
        beta=vae_cfg.beta,
        device=device,
    )
    vae_ckpt = torch.load(diff_cfg.betavae_ckpt_path, map_location=device)
    vae.load_state_dict(vae_ckpt["model_state_dict"])
    vae.eval()
    print(f"beta-VAE loaded (step {vae_ckpt['step']})")
else:
    # Conditional, but no VAE configured -- the diffusion model was trained on precomputed
    # latents (e.g. AlexNet-fc6-PCA, see notebooks/alexnet_pca_latents.ipynb) row-aligned
    # with val_images.npy, so z is looked up from val_latents.npy instead of vae.encode()
    # (see the sampling loop below).
    vae = None
    print("Conditional on precomputed latents -- no VAE to load, z comes from val_latents.npy")

# --- UNet ---
unet = UNet(
    in_channels=diff_cfg.in_channels,
    image_size=diff_cfg.image_size,
    model_channels=diff_cfg.model_channels,
    channel_mult=diff_cfg.channel_mult,
    num_res_blocks=diff_cfg.num_res_blocks,
    attention_resolutions=diff_cfg.attention_resolutions,
    dropout=0.0,
    latent_dim=diff_cfg.latent_dim,
)

if DIFFUSION_CKPT_PATH is None:
    ckpt_dir = os.path.join(diff_cfg.output_dir, "checkpoints")
    DIFFUSION_CKPT_PATH = os.path.join(ckpt_dir, "best_ckpt.pt")
    if not os.path.exists(DIFFUSION_CKPT_PATH):
        raise FileNotFoundError(f"best_ckpt.pt not found in {ckpt_dir}")

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet.load_state_dict(diff_ckpt["model_state_dict"])

if USE_EMA_WEIGHTS and "ema_state_dict" in diff_ckpt:
    from models.ema import ema_shadow_to_model_state_dict
    ema_sd = diff_ckpt["ema_state_dict"]
    unet.load_state_dict(ema_shadow_to_model_state_dict(unet, ema_sd["shadow"]))
    print(f"Sampling from EMA weights (EMA step {ema_sd['step']}, decay {ema_sd['decay']})")
elif USE_EMA_WEIGHTS:
    print("USE_EMA_WEIGHTS=True but checkpoint has no ema_state_dict "
          "(older run, or ema_decay wasn't set during training) -- using raw weights")

unet = unet.to(device).eval()
print(f"UNet loaded: {DIFFUSION_CKPT_PATH}  (step {diff_ckpt['step']})")

## Load reference images

In [ ]:
if diff_cfg.dataset_type == "tiny_imagenet":
    subprocess.run(["wget", "http://cs231n.stanford.edu/tiny-imagenet-200.zip"], check=True)
    subprocess.run(["unzip", "-q", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["rm", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["ls", "/content/tiny-imagenet-200"], check=True)
elif diff_cfg.dataset_type == "imagenet64":
    subprocess.run(["mkdir", "-p", "/content/data/imagenet64/"])
    subprocess.run(["cp", os.path.join(REPO_ROOT, f"data/imagenet64/val_images.npy"), "/content/data/imagenet64/"], check=True)

    # Only stage precomputed latents (e.g. AlexNet-fc6-PCA) for runs that are actually
    # conditioned on them -- same signal as the VAE-loading cell above, so an unconditional
    # or VAE-conditioned imagenet64 run never accidentally picks up (image, latent) pairs.
    if not diff_cfg.unconditional and not hasattr(diff_cfg, 'betavae_config_path'):
        # subprocess.run([
        #     "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/train_latents_z{diff_cfg.latent_dim}.npy"),
        #     diff_cfg.data_dir,
        # ], check=True)
        subprocess.run([
            "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/val_latents_z{diff_cfg.latent_dim}.npy"),
            diff_cfg.data_dir,
        ], check=True)
        print("Staged precomputed latents into data_dir -- conditioning on precomputed latents")
    else:
        print("No precomputed-latents staging needed (unconditional or VAE-conditioned run)")
elif diff_cfg.dataset_type == "stimulus":
    subprocess.run(["mkdir", "-p", "/content/data/"])
    subprocess.run(["cp", os.path.join(REPO_ROOT, f"data/15901Stimuli.zip"), "/content/data/"], check=True)
    # unzip
    subprocess.run(["unzip", "/content/data/15901Stimuli.zip", "-d", "/content/data/15901Stimuli/"])

In [ ]:
# # Unlike inspect_diffusion.ipynb (which only loads real images when LOAD_TEST_IMAGES is
# # set, or a VAE needs something to encode), FID always needs a real reference
# # distribution -- so this always builds the val split via the same dispatcher used at
# # training time, regardless of UNCONDITIONAL.
# _, val_dl = build_dataloaders(diff_cfg)

In [ ]:
# Unlike inspect_diffusion.ipynb (which only loads real images when LOAD_TEST_IMAGES is
# set, or a VAE needs something to encode), FID always needs a real reference
# distribution -- so this always builds the val split, regardless of UNCONDITIONAL.
#
# imagenet64 gets a dedicated val-only loader: build_dataloaders (-> build_imagenet64_
# dataloaders) requires train_images.npy to exist too, which FID never touches and which
# is wasteful to stage/keep around (~15 GB) just to satisfy that check.
if diff_cfg.dataset_type == "imagenet64":
    val_dl = build_imagenet64_val_dataloader(
        data_dir=diff_cfg.data_dir,
        image_size=diff_cfg.image_size,
        batch_size=diff_cfg.batch_size,
        latent_dim=getattr(diff_cfg, "latent_dim", None),
        unconditional=getattr(diff_cfg, "unconditional", False),
    )
elif diff_cfg.dataset_type == "stimulus":
    val_dl = build_encode_dataloader(
        data_dir="/content/data/15901Stimuli/",
        image_size=diff_cfg.image_size,
        batch_size=diff_cfg.batch_size,
        num_workers=0
    )
else:
    _, val_dl = build_dataloaders(diff_cfg)

real_batches = []
n_real = 0
for batch in val_dl:
    imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
    real_batches.append(imgs)
    n_real += imgs.shape[0]
    if n_real >= N_FID_SAMPLES:
        break
real_images = torch.cat(real_batches, dim=0)[:N_FID_SAMPLES]

if real_images.shape[0] < N_FID_SAMPLES:
    print(f"Real reference: only {real_images.shape[0]} images available in the val split "
          f"(< N_FID_SAMPLES={N_FID_SAMPLES}) -- using the full val split rather than "
          f"duplicating real images to pad it out.")
else:
    print(f"Real reference: {real_images.shape[0]} images (capped at N_FID_SAMPLES)")

### Noise schedule

In [ ]:
np_type = getattr(diff_cfg, 'noise_process', 'ddpm')
embed_scale = getattr(diff_cfg, 'num_timesteps', 1000)

if np_type == 'ddpm':
    noise_proc = DDPM(
        num_timesteps=diff_cfg.num_timesteps,
        beta_schedule=diff_cfg.beta_schedule,
        min_snr_gamma=getattr(diff_cfg, 'min_snr_gamma', 5.0),
        device=device
    )
    schedule = make_noise_schedule(diff_cfg, device)
    print(f"DDPM  T={diff_cfg.num_timesteps}  beta_0={schedule['betas'][0]:.5f}  beta_T={schedule['betas'][-1]:.4f}")

elif np_type == 'flow':
    noise_proc = RectifiedFlow()
    schedule   = None
    print("RectifiedFlow -- continuous ODE, t in [0, 1]")

elif np_type == 'vpsde':
    noise_proc = VPSDE(
        beta_min=getattr(diff_cfg, 'vpsde_beta_min', 0.01),
        beta_max=getattr(diff_cfg, 'vpsde_beta_max', 5.0),
    )
    schedule   = None
    print(f"VPSDE  beta_min={noise_proc.beta_min}  beta_max={noise_proc.beta_max}")

else:
    raise ValueError(f"Unknown noise_process: {np_type!r}")

print(f"embed_scale={embed_scale}")

## Generate samples in a loop (for FID)

In [ ]:
generated_images_path = os.path.join(diff_cfg.output_dir, "visualizations", "generated_images.npy")
os.makedirs(os.path.dirname(generated_images_path), exist_ok=True)

# Load whatever's cached, if any -- a shortfall (e.g. a crashed previous run) only tops up
# the remainder instead of regenerating from scratch.
if os.path.exists(generated_images_path):
    cached = np.load(generated_images_path)
    generated_batches = [torch.from_numpy(cached)]
    n_generated = cached.shape[0]
    print(f"Loaded {n_generated} cached generated images from {generated_images_path}")
else:
    generated_batches = []
    n_generated = 0

if n_generated >= N_FID_SAMPLES:
    generated_images = torch.cat(generated_batches, dim=0)[:N_FID_SAMPLES]
    print(f"Cache already has >= N_FID_SAMPLES={N_FID_SAMPLES} images -- nothing to generate")
else:
    print(f"Generating {N_FID_SAMPLES - n_generated} more images "
          f"({n_generated} cached, target {N_FID_SAMPLES})")

    # Real images (and, for conditional models, their matching latents) also serve as the
    # source of z for generation, cycled with itertools.cycle if N_FID_SAMPLES exceeds the
    # val split size. Cycling only affects z variety here -- sampling noise still makes each
    # generated image distinct even when the same conditioning z repeats.
    z_source = itertools.cycle(val_dl) if not UNCONDITIONAL else None

    # Checkpoint to disk every ~5% of N_FID_SAMPLES so a crash mid-run only costs that much
    # progress, not the whole thing.
    checkpoint_every = max(1, round(N_FID_SAMPLES * 0.05))
    n_since_checkpoint = 0

    with torch.no_grad():
        pbar = tqdm(total=N_FID_SAMPLES, initial=n_generated, desc="Sampling for FID")
        while n_generated < N_FID_SAMPLES:
            b = min(FID_BATCH_SIZE, N_FID_SAMPLES - n_generated)

            if UNCONDITIONAL:
                z_batch = torch.zeros(b, diff_cfg.latent_dim, device=device)
            else:
                batch = next(z_source)
                src_imgs, src_latents = batch if isinstance(batch, (list, tuple)) else (batch, None)
                src_imgs = src_imgs[:b].to(device)
                z_batch = vae.encode(src_imgs)[0] if vae is not None else src_latents[:b].to(device)

            if np_type == 'ddpm':
                fake = diffusion_sample(
                    z_batch, unet, schedule, diff_cfg, device,
                    sampler=SAMPLER, eta=ETA, num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                )
            elif np_type == 'flow':
                fake = flow_sample(
                    z_batch, unet, diff_cfg, device,
                    num_steps=NUM_INFERENCE_STEPS, guidance_scale=GUIDANCE_SCALE,
                )
            elif np_type == 'vpsde':
                fake = vpsde_sample(
                    z_batch, unet, diff_cfg, device,
                    num_steps=NUM_INFERENCE_STEPS, guidance_scale=GUIDANCE_SCALE,
                )

            generated_batches.append(fake.cpu())
            n_generated += fake.shape[0]
            n_since_checkpoint += fake.shape[0]
            pbar.update(fake.shape[0])

            if n_since_checkpoint >= checkpoint_every:
                np.save(generated_images_path, torch.cat(generated_batches, dim=0).numpy())
                pbar.write(f"Checkpointed {n_generated}/{N_FID_SAMPLES} images to {generated_images_path}")
                n_since_checkpoint = 0
        pbar.close()

    generated_images = torch.cat(generated_batches, dim=0)[:N_FID_SAMPLES]
    np.save(generated_images_path, generated_images.numpy())
    print(f"Generated {generated_images.shape[0]} total images, saved to {generated_images_path}")

### Sanity check

In [ ]:
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 2, 4.5), squeeze=False)

def _show(ax, img):
    img = img.permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img.squeeze(-1) if img.shape[-1] == 1 else img, cmap='gray' if img.shape[-1] == 1 else None)
    ax.axis('off')

for col in range(n_show):
    _show(axes[0, col], real_images[col])
    _show(axes[1, col], generated_images[col])

axes[0, 0].set_ylabel("real", fontsize=9, rotation=0, ha='right', va='center', labelpad=30)
axes[1, 0].set_ylabel("generated", fontsize=9, rotation=0, ha='right', va='center', labelpad=30)
fig.suptitle("Real vs. generated -- eyeball before trusting the FID number below", fontsize=10)
plt.tight_layout()
plt.show()

## Compute FID

In [ ]:
# Standard torchvision Inception-v3 (ImageNet-pretrained) pool3 features (2048-dim),
# obtained by replacing the final fc with Identity. This is not bit-identical to the
# original FID paper's TF-ported Inception checkpoint (pytorch-fid / torch-fidelity vendor
# that one for cross-paper comparability) -- fine here since we only need a consistent
# metric to compare our own pretrained vs. finetuned checkpoints against each other.
inception = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
inception.fc = torch.nn.Identity()
inception = inception.eval().to(device)

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
_IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def get_inception_features(images: torch.Tensor, batch_size: int = 64) -> np.ndarray:
    """images: [N, C, H, W] float in [0, 1]. Returns [N, 2048] numpy array."""
    feats = []
    for i in tqdm(range(0, images.shape[0], batch_size), desc="Inception features", leave=False):
        batch = images[i:i + batch_size].to(device)
        if batch.shape[1] == 1:
            batch = batch.repeat(1, 3, 1, 1)
        batch = F.interpolate(batch, size=(299, 299), mode='bilinear', align_corners=False)
        batch = (batch - _IMAGENET_MEAN) / _IMAGENET_STD
        feats.append(inception(batch).cpu().numpy())
    return np.concatenate(feats, axis=0)

In [ ]:
def frechet_distance(feat_real: np.ndarray, feat_fake: np.ndarray, eps: float = 1e-6) -> float:
    mu_r, mu_f = feat_real.mean(axis=0), feat_fake.mean(axis=0)
    sigma_r = np.cov(feat_real, rowvar=False)
    sigma_f = np.cov(feat_fake, rowvar=False)

    diff = mu_r - mu_f
    covmean, _ = linalg.sqrtm(sigma_r @ sigma_f, disp=False)
    if not np.isfinite(covmean).all():
        # near-singular covariance (small sample count) -- nudge with eps*I, the standard
        # pytorch-fid workaround
        offset = np.eye(sigma_r.shape[0]) * eps
        covmean = linalg.sqrtm((sigma_r + offset) @ (sigma_f + offset))
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return float(diff @ diff + np.trace(sigma_r + sigma_f - 2 * covmean))


feat_real = get_inception_features(real_images)
feat_fake = get_inception_features(generated_images)
fid_score = frechet_distance(feat_real, feat_fake)

print(f"FID ({diff_cfg.experiment_name}): {fid_score:.3f}")
print(f"  real: {feat_real.shape[0]} images | generated: {feat_fake.shape[0]} images")